# B3 — mixed displacement-stress formulation: production-scale diagnostic

**Follow-up paper, item 1 of Professor Rabczuk's 2026-10-16 email** ("test in this order a few things" — this is the first): the network predicts both `u` and the first Piola-Kirchhoff stress `P`, trained against two physics residuals — constitutive consistency (`P = dpsi/dF`) and equilibrium (`Div_0(P) = 0`) — instead of total potential energy. Kept completely separate from `train_B3.py` (which produced the real, already-submitted B3 numbers), in a new `mixed_stress_B3.py` / `train_B3_mixed_stress.py`.

**This is NOT a training run.** Everything in this cell runs on a FIXED batch (same material/load sample every iteration, no new sampling noise), for only a few hundred iterations, at the real production model size — the exact same methodology that originally found `train_B3.py`'s own `OUTPUT_SCALE=0.02` (see that constant's docstring).

**Why this check matters before any real training run**: the same diagnostic, run on CPU with a tiny toy model (32 hidden, 2 layers) this session, found the naive `w_constitutive=w_equilibrium=1.0` loss is stable (no divergence) but ~99.995% dominated by the constitutive term (a ~28,000x imbalance from iteration 1) — equilibrium's gradient contribution was noise-level and actually drifted slightly worse while constitutive improved. Rebalancing to `w_equilibrium=28000` fixed this on the toy model (equilibrium converges to ~0 within ~20 iterations, constitutive still improves, total loss stays stable). That ratio was measured on a tiny, undertrained model — **this cell re-measures it at real production model size on GPU**, which is the only way to know if 28000 is still the right order of magnitude before trusting a long run.

**What to watch for in the printed output**: (1) the loss should be stable — no NaN, no runaway growth; (2) BOTH `constitutive` and `equilibrium` should visibly decrease over the run, not just whichever one is larger. If `equilibrium` is still flat/ignored at `w_equilibrium=28000`, that means production scale needs a different ratio — send the full printed output back rather than guessing a new number.

In [ ]:
# =====================================================================
#  CELL -- B3 mixed displacement-stress: PRODUCTION-SCALE diagnostic
#  (fixed batch, ~300 iterations, NOT a training run). See the markdown
#  cell above for what this checks and why.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

# First, re-run the exact 5 CPU correctness checks here too -- cheap
# (a few seconds), and confirms the environment's own torch/torchfem
# versions reproduce the same guarantees before trusting anything built
# on top of them.
from omar_pfem.test_mixed_stress_B3 import (
    test_constitutive_matches_autograd, test_divergence_zero_for_uniform_field,
    test_divergence_matches_known_nonzero_value, test_affine_field_end_to_end,
    test_loss_is_finite_and_differentiable,
)
print('\n=== Re-running the 5 CPU correctness checks in this environment ===')
test_constitutive_matches_autograd()
test_divergence_zero_for_uniform_field()
test_divergence_matches_known_nonzero_value()
test_affine_field_end_to_end()
test_loss_is_finite_and_differentiable()
print('All 5 checks passed in this environment too.\n')

# Now the real diagnostic: PRODUCTION model size (train_B3_mixed_stress's
# own defaults: n_hidden=256, n_layers=4), fixed batch, 300 iterations,
# on GPU. Not a training run -- no checkpoint is meant to be kept from
# this (none is saved; this cell never calls train()).
print('=== Production-scale fixed-batch diagnostic (300 iterations) ===')
from omar_pfem.diagnose_mixed_stress_B3_scale import main as diagnose_main
diagnose_main(['--n_steps', '300', '--batch_size', '8'])

print("\nSend this cell's full printed output back -- need to see whether "
      "w_equilibrium=28000 still balances the two terms at production model "
      "size, or whether it needs adjusting before any real training run.")
